# Agents Workshop: Build an SOW Drafting Agent in AI Playground

Attendees build a simple agent in AI Playground that drafts a new Statement of Work (SOW) for a client, based on Northgate's signed SOWs. They see the four parts every agent is made of, compare the drafts from different models, and turn the agent into an app without writing code.

## What is an agent?

An agent isn't a single model. It's a small system made of four parts:

| Part | What it is | In this notebook |
| --- | --- | --- |
| **Governed enterprise data** | The facts the agent is allowed to use, protected by Unity Catalog permissions | The tables behind your Genie Agent: contract terms from the signed SOWs, agreed and standard day rates, engagements |
| **An LLM** | The model that reads, reasons and writes | Any tools-enabled model in Playground, and you can swap it |
| **Logic** | The rules: instructions and code | The system prompt, plus the instructions and example SQL inside the Genie Agent |
| **Tool calls** | Actions the model can take, including calling other agents | The Genie Agent you built in [1. Genie Agents](#notebook-3452153026155247), used as a tool |

The model decides **when** to call a tool and **what to ask** it. The Genie Agent decides **what** data it can see, with the user's own permissions.

**The scenario.** An engagement manager types: *"I have a new client I need to draft an SOW for, can you create me the same type of SOW?"* The agent asks the Genie Agent for the terms of a similar signed SOW, prices the team at today's standard rates and writes a draft for the partner to review.

| Part | Topic | Format |
| --- | --- | --- |
| 1 | Build the agent in AI Playground | Hands-on |
| 2 | Swap models and compare the drafts | Hands-on |
| 3 | Turn the agent into an app | Hands-on |

**Prerequisites:** you've built the Genie Agent in [1. Genie Agents](#notebook-3452153026155247) and added the contract tables (`contract_terms`, `contracted_rates`) to it in Part 1B of [2. Document Intelligence](#notebook-3452153026155248). Part 3 needs Databricks Apps and the **Managed MCP Servers** preview enabled in the workspace.

## Part 1: Build the agent in AI Playground (hands-on)

1. Click **Playground** in the left navigation. Choose a model marked **Tools enabled**, for example `databricks-claude-sonnet-4-6`.
2. Click **Tools > + Add tool**, choose **Genie Agent**, and select the Genie Agent you built in [1. Genie Agents](#notebook-3452153026155247). That's the agent's only tool, and it's another agent.
3. Paste this **system prompt**. This is the agent's logic:
```
You are the Northgate Advisory SOW drafting assistant for engagement managers.
- To draft a new SOW, first ask the Genie Agent for the contract terms of a signed SOW of the same type:
  billing basis, fee or fee cap, payment terms, expense policy and the agreed day rates by grade.
- Price the new team at Northgate's standard day rates (ask the Genie Agent for the grade rate card),
  not the agreed rates of the old SOW, which may be discounted for that client.
- Write the draft with these sections: 1. Parties, 2. Scope of services, 3. Term, 4. Commercial terms,
  5. Agreed day rates, 6. Expenses, 7. Payment terms, 8. Signatures.
- Ask for anything you need about the new client: client name, scope, dates, team and billing basis.
  Mark anything still missing as [PLACEHOLDER]. Never copy the old client's name, PO number or signatories.
- Say which SOW you used as the template. Money is GBP. Never invent a number.
- The result is a draft for the engagement partner to review, not a signed contract.
```
4. Ask: *I have a new client I need to draft an SOW for, can you create me the same type of SOW as our Cyber Maturity Assessment?*
5. Answer its questions, for example: *The client is Brightwater Retail Group, a UK retailer. Start on 11 January 2027 for 8 weeks, capped T&M, with a Director one day a week, a Manager full-time and two Consultants full-time.*
6. Open the **trace** under the answer and find each part of the agent:

| What you see in the trace | Part of the agent |
| --- | --- |
| The questions the model sent to the Genie Agent | Tool calls |
| The SQL Genie wrote and the rows it returned | Governed enterprise data |
| The rules from the system prompt it followed (sections, standard rates, placeholders) | Logic |
| The draft itself | LLM |

7. Test the edges:
   * *Draft an SOW like the one we did for a tax client.* There are several, so it should list them or ask which one.
   * *Base it on SOW-26-099.* That SOW doesn't exist, so it should say so rather than make one up.

## Part 2: Swap models and compare the drafts (hands-on)

The data, the Genie Agent tool and the system prompt stay the same. Only the LLM changes.

1. Click **+** at the top of Playground to open a second panel side by side, and choose a different tools-enabled model (for example a GPT, Gemini or Llama model).
2. Turn on **Sync** so both panels get the same system prompt, tools and messages.
3. Send the Part 1 request and follow-up again, and compare:
   * Did both pick the same SOW as the template?
   * Did they ask for missing details, or guess?
   * Did they use the standard day rates, and is the arithmetic right?
   * Which draft reads better? Which is shorter, faster or uses fewer tokens?
4. Try a smaller, cheaper model in one panel. Is its draft good enough for a first version?
5. Change one line of the system prompt (for example *"Write in plain English for a non-specialist client"*) and see how each model responds.

**Discussion:** switching models is a dropdown, not a rebuild, because the data and the Genie Agent live in Unity Catalog, not in the model. In production you'd compare models with an MLflow evaluation set rather than by eye.

## Part 3: Turn the agent into an app (hands-on)

Playground has a built-in export that turns your prototype into a Databricks App with a chat UI, using the same model, system prompt and tools you tested.

1. Choose the panel with the model you liked best, then click **Get code > Export to Databricks Apps**.
2. Fill in the dialog:
   * **App Name:** must start with `agent-` and use only lowercase letters, numbers and hyphens, for example `agent-<initials>-sow-drafter`.
   * **App Description:** *Drafts new Northgate SOWs from the terms of signed SOWs and standard day rates.*
   * **MLflow Experiment:** create a new one. The app records its traces there.
3. Click **Export**. Databricks installs an agent app template, grants the app access to the model, the MLflow experiment and your Genie Agent, and generates the agent code from your Playground settings.
4. When the success dialog appears, click **View Agent** and ask the Part 1 request again in the app's chat UI.
5. Share the app with a colleague from the app's **Permissions** settings. They also need access to the Genie Agent and its tables to get answers.

**Facilitator:** check before the session that Databricks Apps and the Managed MCP Servers preview are enabled. Without them, the export isn't available.

## Wrap-up

* **An agent is a system, not a model:** governed data, an LLM, logic (prompt and instructions) and tool calls.
* **Agents can be tools for other agents.** The Genie Agent you curated in [1. Genie Agents](#notebook-3452153026155247) is the governed data layer; the Playground agent adds the drafting.
* **The model is swappable:** a dropdown in Playground, with the same tool and prompt.
* **A prototype becomes an app in a few clicks,** with the same configuration you tested.